In [9]:
import os, json

XAI_BASE = '/kaggle/input/datasets/michaelmarkey64/kg-xai-0206/'

print("=== XAI Dataset files ===")
for f in sorted(os.listdir(XAI_BASE)):
    size = os.path.getsize(os.path.join(XAI_BASE, f))
    print(f"  {f}  ({size:,} bytes)")

with open(XAI_BASE + 'wilcoxon_results.json') as f:
    wilcoxon = json.load(f)
print(f"\nWilcoxon keys: {list(wilcoxon.keys())}")
print(f"Seeds: {wilcoxon.get('seeds', 'not found')}")

=== XAI Dataset files ===
  gaBERT_KG_CRF_MWE_phase3.conll  (61,791 bytes)
  gaBERT_KG_CRF_run1_mixed.conll  (45,479 bytes)
  gaBERT_KG_CRF_seed42.conll  (45,463 bytes)
  gaBERT_RDA_CRF.conll  (61,679 bytes)
  gaBERT_RDA_CRF_MWE.conll  (40,327 bytes)
  gaBERT_RDA_CRF_seed42.conll  (45,515 bytes)
  wilcoxon_results.json  (955 bytes)

Wilcoxon keys: ['seeds', 'baseline_f1s', 'kg_f1s', 'baseline_mean', 'baseline_std', 'kg_mean', 'kg_std', 'wilcoxon_stat', 'p_value', 'effect_size_r', 'z_score', 'ci_baseline', 'ci_kg', 'ci_diff', 'significant']
Seeds: [42, 123, 456, 789, 1234, 5678, 9999]


In [10]:
!pip install conlleval -q
from conlleval import evaluate
import pandas as pd

files = {
    'Baseline (RDA) Mixed':  XAI_BASE + 'gaBERT_RDA_CRF.conll',
    'Baseline (RDA) MWE':    XAI_BASE + 'gaBERT_RDA_CRF_MWE.conll',
    'KG Run 3 MWE':          XAI_BASE + 'gaBERT_KG_CRF_MWE_phase3.conll',
    'Baseline seed42':       XAI_BASE + 'gaBERT_RDA_CRF_seed42.conll',
    'KG seed42':             XAI_BASE + 'gaBERT_KG_CRF_seed42.conll',
}

rows = []
for name, path in files.items():
    with open(path) as f:
        lines = f.read().splitlines()
    r = evaluate(lines)
    overall = r['overall']['chunks']['evals']
    slots   = r['slots']['chunks']
    rows.append({
        'Run':        name,
        'Overall F1': round(overall['f1'], 4),
        'Overall P':  round(overall['prec'], 4),
        'Overall R':  round(overall['rec'], 4),
        'PER F1':     round(slots['PER']['evals']['f1'], 4),
        'LOC F1':     round(slots['LOC']['evals']['f1'], 4),
        'ORG F1':     round(slots['ORG']['evals']['f1'], 4),
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

                 Run  Overall F1  Overall P  Overall R  PER F1  LOC F1  ORG F1
Baseline (RDA) Mixed      0.7745     0.7587     0.7910  0.8201  0.8070  0.6787
  Baseline (RDA) MWE      0.7540     0.7172     0.7948  0.8219  0.7933  0.6228
        KG Run 3 MWE      0.7446     0.7174     0.7740  0.7988  0.7712  0.6625
     Baseline seed42      0.7673     0.7546     0.7804  0.8634  0.7566  0.6735
           KG seed42      0.7725     0.7668     0.7783  0.8521  0.7552  0.6940


In [11]:
def load_conll(path):
    tokens, trues, preds = [], [], []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                tokens.append(parts[0])
                trues.append(parts[2])
                preds.append(parts[3])
    return tokens, trues, preds

_, trues_bl, preds_bl = load_conll(XAI_BASE + 'gaBERT_RDA_CRF_seed42.conll')
_, trues_kg, preds_kg = load_conll(XAI_BASE + 'gaBERT_KG_CRF_seed42.conll')

assert trues_bl == trues_kg, "Gold labels don't match — files are misaligned"

fixes, errors, same_correct, same_wrong = [], [], [], []

for i, (true, bl, kg) in enumerate(zip(trues_bl, preds_bl, preds_kg)):
    if bl == kg:
        if bl == true:
            same_correct.append(i)
        else:
            same_wrong.append(i)
    elif bl != true and kg == true:
        fixes.append((i, true, bl, kg))
    elif bl == true and kg != true:
        errors.append((i, true, bl, kg))

print(f"Total tokens compared: {len(trues_bl)}")
print(f"Same and correct:      {len(same_correct)}")
print(f"Same and wrong:        {len(same_wrong)}")
print(f"KG fixes baseline:     {len(fixes)}")
print(f"KG breaks baseline:    {len(errors)}")
print(f"\nNet token-level change: {len(fixes) - len(errors):+d}")

# Per-class breakdown of fixes and errors
for label in ['B-PER','I-PER','B-LOC','I-LOC','B-ORG','I-ORG']:
    f = sum(1 for _, t, _, _ in fixes  if t == label)
    e = sum(1 for _, t, _, _ in errors if t == label)
    if f > 0 or e > 0:
        print(f"  {label}: fixes={f}  errors={e}  net={f-e:+d}")

Total tokens compared: 4704
Same and correct:      4431
Same and wrong:        170
KG fixes baseline:     35
KG breaks baseline:    44

Net token-level change: -9
  B-PER: fixes=7  errors=0  net=+7
  I-PER: fixes=3  errors=6  net=-3
  B-LOC: fixes=2  errors=6  net=-4
  I-LOC: fixes=7  errors=1  net=+6
  B-ORG: fixes=3  errors=13  net=-10
  I-ORG: fixes=1  errors=8  net=-7


In [12]:
import pickle
import pandas as pd

KG_BASE = '/kaggle/input/datasets/michaelmarkey64/kg-embeddings-baseline-dissertation/'

# Load embeddings
with open(KG_BASE + 'TransE_qid_embeddings.pkl', 'rb') as f:
    qid_to_embedding = pickle.load(f)

# Load node CSVs
per  = pd.read_csv(KG_BASE + 'per_nodes.csv')
loc  = pd.read_csv(KG_BASE + 'loc_nodes.csv')
org  = pd.read_csv(KG_BASE + 'org_nodes.csv')

# Build surface_to_qid
surface_to_qid = {}

for _, row in loc.iterrows():
    for col in ['entity', 'canonical', 'label_ga', 'label_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[row[col].strip()] = row['qid']

for _, row in org.iterrows():
    for col in ['entity', 'canonical', 'label_ga', 'label_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[row[col].strip()] = row['qid']

for _, row in per.iterrows():
    for col in ['label_ga', 'label_en', 'full_name_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[row[col].strip()] = row['qid']

print(f"surface_to_qid entries: {len(surface_to_qid)}")
print(f"qid_to_embedding entries: {len(qid_to_embedding)}")

surface_to_qid entries: 2278
qid_to_embedding entries: 1563


In [13]:
def read_conll_file(file_path):
    data = []
    current_sentence = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line.startswith('-DOCSTART-'):
                continue
            if line:
                parts = line.split()
                word = parts[0]
                ner_label = parts[-1]
                current_sentence.append((word, ner_label))
            else:
                if current_sentence:
                    data.append(current_sentence)
                    current_sentence = []
    if current_sentence:
        data.append(current_sentence)
    return data

ADKINS_BASE = '/kaggle/input/datasets/michaelmarkey64/adkins-et-al-2025/'
test_data = read_conll_file(ADKINS_BASE + 'NER_Irish_test.conll')

# Extract all gold entities with their surface forms
def extract_entities(sentences):
    entities = []
    for sent in sentences:
        i = 0
        while i < len(sent):
            word, label = sent[i]
            if label.startswith('B-'):
                etype = label[2:]
                span = [word]
                j = i + 1
                while j < len(sent) and sent[j][1] == f'I-{etype}':
                    span.append(sent[j][0])
                    j += 1
                surface = ' '.join(span)
                qid = surface_to_qid.get(surface)
                has_embedding = qid is not None and qid in qid_to_embedding
                entities.append({
                    'surface': surface,
                    'type': etype,
                    'qid': qid,
                    'covered': has_embedding
                })
                i = j
            else:
                i += 1
    return entities

entities = extract_entities(test_data)
df_ents = pd.DataFrame(entities)

print(f"Total gold entities: {len(df_ents)}")
print(f"KG covered:          {df_ents['covered'].sum()} ({df_ents['covered'].mean()*100:.1f}%)")
print(f"Not covered:         {(~df_ents['covered']).sum()}")
print(f"\nCoverage by type:")
print(df_ents.groupby('type')['covered'].agg(['sum','count','mean']).round(3))

Total gold entities: 335
KG covered:          70 (20.9%)
Not covered:         265

Coverage by type:
      sum  count   mean
type                   
LOC    49    132  0.371
ORG    14     95  0.147
PER     7    108  0.065


In [14]:
import pandas as pd
import numpy as np

def analyze_entities_robust(conll_path):
    """
    Parses a single CoNLL file sentence by sentence to perfectly match 
    extracted entity spans with their coverage and correctness.
    """
    results = []
    current_words = []
    current_trues = []
    current_preds = []
    
    # 1. Parse file into sentence blocks natively
    with open(conll_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('-DOCSTART-') or not line:
                if current_words:
                    # Process the completed sentence
                    results.extend(extract_spans_from_sent(current_words, current_trues, current_preds))
                    current_words, current_trues, current_preds = [], [], []
            else:
                parts = line.split()
                current_words.append(parts[0])
                current_trues.append(parts[2])
                current_preds.append(parts[3])
                
        if current_words:
            results.extend(extract_spans_from_sent(current_words, current_trues, current_preds))
            
    return pd.DataFrame(results)

def extract_spans_from_sent(words, trues, preds):
    """Helper to extract named entities chunked by B- tags from a single sentence"""
    sent_results = []
    i = 0
    while i < len(words):
        true_label = trues[i]
        if true_label.startswith('B-'):
            etype = true_label[2:]
            span_words = [words[i]]
            span_true = [true_label]
            span_pred = [preds[i]]
            
            j = i + 1
            while j < len(words) and trues[j] == f'I-{etype}':
                span_words.append(words[j])
                span_true.append(trues[j])
                span_pred.append(preds[j])
                j += 1
                
            surface = ' '.join(span_words).strip()
            qid = surface_to_qid.get(surface)
            covered = qid is not None and qid in qid_to_embedding
            correct = span_true == span_pred
            
            sent_results.append({
                'surface': surface,
                'type': etype,
                'covered': covered,
                'true': span_true,
                'pred': span_pred,
                'correct': correct
            })
            i = j
        else:
            i += 1
    return sent_results

# Regenerate DataFrames cleanly
df_bl = analyze_entities_robust(XAI_BASE + 'gaBERT_RDA_CRF_seed42.conll')
df_kg = analyze_entities_robust(XAI_BASE + 'gaBERT_KG_CRF_seed42.conll')

In [15]:
print("=== 1. Prediction Change Diff (Token-level) ===")
# Reuse your previous token arrays logic safely
_, trues_bl, preds_bl = load_conll(XAI_BASE + 'gaBERT_RDA_CRF_seed42.conll')
_, _, preds_kg = load_conll(XAI_BASE + 'gaBERT_KG_CRF_seed42.conll')

fixes, errors = 0, 0
for t, b, k in zip(trues_bl, preds_bl, preds_kg):
    if b != t and k == t:
        fixes += 1
    elif b == t and k != t:
        errors += 1

print(f"Total KG Fixes (Error -> Correct): {fixes}")
print(f"Total KG Errors (Correct -> Error): {errors}")
print(f"Net Token Improvement: {fixes - errors:+d}\n")

=== 1. Prediction Change Diff (Token-level) ===
Total KG Fixes (Error -> Correct): 35
Total KG Errors (Correct -> Error): 44
Net Token Improvement: -9



In [16]:
print("=== 2. Per-Class F1 Analysis Table ===")
# Uses the table output parsed cleanly from df_results in Cell 9
print(df_results[['Run', 'Overall F1', 'PER F1', 'LOC F1', 'ORG F1']].to_string(index=False))
print("\n")

=== 2. Per-Class F1 Analysis Table ===
                 Run  Overall F1  PER F1  LOC F1  ORG F1
Baseline (RDA) Mixed      0.7745  0.8201  0.8070  0.6787
  Baseline (RDA) MWE      0.7540  0.8219  0.7933  0.6228
        KG Run 3 MWE      0.7446  0.7988  0.7712  0.6625
     Baseline seed42      0.7673  0.8634  0.7566  0.6735
           KG seed42      0.7725  0.8521  0.7552  0.6940




In [17]:
print("=== 3. Coverage-Stratified Analysis (Entity-level) ===")
for covered in [True, False]:
    label = "Covered" if covered else "Uncovered"
    
    bl_sub = df_bl[df_bl['covered'] == covered]
    kg_sub = df_kg[df_kg['covered'] == covered]
    
    bl_acc = bl_sub['correct'].mean() if len(bl_sub) > 0 else 0
    kg_acc = kg_sub['correct'].mean() if len(kg_sub) > 0 else 0
    
    print(f"{label} Entities (n={len(bl_sub)}):")
    print(f"  Baseline Accuracy : {bl_acc:.3f}")
    print(f"  KG Accuracy       : {kg_acc:.3f}")
    print(f"  Accuracy Delta    : {kg_acc - bl_acc:+.3f}\n")

=== 3. Coverage-Stratified Analysis (Entity-level) ===
Covered Entities (n=0):
  Baseline Accuracy : 0.000
  KG Accuracy       : 0.000
  Accuracy Delta    : +0.000

Uncovered Entities (n=469):
  Baseline Accuracy : 0.797
  KG Accuracy       : 0.785
  Accuracy Delta    : -0.013



In [18]:
# ==========================================
# 1. REBUILD DICTIONARY WITH LOWERCASE KEYS
# ==========================================
surface_to_qid = {}

for _, row in loc.iterrows():
    for col in ['entity', 'canonical', 'label_ga', 'label_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[str(row[col]).strip().lower()] = row['qid']

for _, row in org.iterrows():
    for col in ['entity', 'canonical', 'label_ga', 'label_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[str(row[col]).strip().lower()] = row['qid']

for _, row in per.iterrows():
    for col in ['label_ga', 'label_en', 'full_name_en']:
        if col in row and pd.notna(row[col]):
            surface_to_qid[str(row[col]).strip().lower()] = row['qid']


# ==========================================
# 2. LABELS PARSER & TEST DATA ALIGNER
# ==========================================
def load_labels_only(conll_path):
    """Extracts true and predicted labels per sentence block from CoNLL file."""
    sentences_labels = []
    current_sent = []
    with open(conll_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('-DOCSTART-') or not line:
                if current_sent:
                    sentences_labels.append(current_sent)
                    current_sent = []
            else:
                parts = line.split()
                # Store (true_label, pred_label)
                current_sent.append((parts[2], parts[3]))
        if current_sent:
            sentences_labels.append(current_sent)
    return sentences_labels


def extract_aligned_entities(test_data, labels_data):
    """Aligns real Irish words from test_data with predicted labels from CoNLL file."""
    results = []
    
    # Iterate through sentences simultaneously 
    for sent_words, sent_labels in zip(test_data, labels_data):
        # Double-check alignment lengths
        min_len = min(len(sent_words), len(sent_labels))
        
        i = 0
        while i < min_len:
            # sent_words format is (word, label) from cell 15 logic
            word = sent_words[i][0] 
            true_label, pred_label = sent_labels[i]
            
            if true_label.startswith('B-'):
                etype = true_label[2:]
                span_words = [word]
                span_true = [true_label]
                span_pred = [pred_label]
                
                j = i + 1
                while j < min_len and sent_labels[j][0] == f'I-{etype}':
                    span_words.append(sent_words[j][0])
                    span_true.append(sent_labels[j][0])
                    span_pred.append(sent_labels[j][1])
                    j += 1
                    
                surface = ' '.join(span_words).strip()
                
                # Dictionary check using the real words
                qid = surface_to_qid.get(surface.lower())
                covered = qid is not None and qid in qid_to_embedding
                correct = span_true == span_pred
                
                results.append({
                    'surface': surface,
                    'type': etype,
                    'covered': covered,
                    'correct': correct
                })
                i = j
            else:
                i += 1
                
    return pd.DataFrame(results)


# ==========================================
# 3. REGENERATE DATAFRAMES AND PRINT RESULTS
# ==========================================
# Load sentence arrays of labels
labels_bl = load_labels_only(XAI_BASE + 'gaBERT_RDA_CRF_seed42.conll')
labels_kg = load_labels_only(XAI_BASE + 'gaBERT_KG_CRF_seed42.conll')

# Build evaluation datasets combining real test words and predictions
df_bl = extract_aligned_entities(test_data, labels_bl)
df_kg = extract_aligned_entities(test_data, labels_kg)

print("=== 3. Coverage-Stratified Analysis (Entity-level) ===")
for covered in [True, False]:
    label = "Covered" if covered else "Uncovered"
    
    bl_sub = df_bl[df_bl['covered'] == covered]
    kg_sub = df_kg[df_kg['covered'] == covered]
    
    bl_acc = bl_sub['correct'].mean() if len(bl_sub) > 0 else 0
    kg_acc = kg_sub['correct'].mean() if len(kg_sub) > 0 else 0
    
    print(f"{label} Entities (n={len(bl_sub)}):")
    print(f"  Baseline Accuracy : {bl_acc:.3f}")
    print(f"  KG Accuracy       : {kg_acc:.3f}")
    print(f"  Accuracy Delta    : {kg_acc - bl_acc:+.3f}\n")

=== 3. Coverage-Stratified Analysis (Entity-level) ===
Covered Entities (n=25):
  Baseline Accuracy : 0.920
  KG Accuracy       : 0.840
  Accuracy Delta    : -0.080

Uncovered Entities (n=388):
  Baseline Accuracy : 0.804
  KG Accuracy       : 0.791
  Accuracy Delta    : -0.013



In [20]:
print("=== Coverage-Stratified Accuracy by Entity Type ===\n")
print(f"{'Group':<30} {'Baseline':>10} {'KG':>10} {'Delta':>8} {'n':>6}")
print("-" * 68)
for etype in ['PER', 'LOC', 'ORG']:
    for covered in [True, False]:
        label = f"{etype} {'covered' if covered else 'uncovered'}"
        bl_sub = df_bl[(df_bl['type']==etype) & (df_bl['covered']==covered)]
        kg_sub = df_kg[(df_kg['type']==etype) & (df_kg['covered']==covered)]
        if len(bl_sub) == 0:
            continue
        bl_acc = bl_sub['correct'].mean()
        kg_acc = kg_sub['correct'].mean()
        print(f"{label:<30} {bl_acc:>10.3f} {kg_acc:>10.3f} {kg_acc-bl_acc:>+8.3f} {len(bl_sub):>6}")

=== Coverage-Stratified Accuracy by Entity Type ===

Group                            Baseline         KG    Delta      n
--------------------------------------------------------------------
PER covered                         1.000      0.750   -0.250      4
PER uncovered                       0.853      0.881   +0.028    143
LOC covered                         0.875      0.812   -0.062     16
LOC uncovered                       0.794      0.794   +0.000    136
ORG covered                         1.000      1.000   +0.000      5
ORG uncovered                       0.752      0.670   -0.083    109


In [22]:
print("=" * 65)
print("XAI ANALYSIS SUMMARY")
print("=" * 65)

bl_mean = wilcoxon['baseline_mean']
kg_mean = wilcoxon['kg_mean']
diff = kg_mean - bl_mean
p = wilcoxon['p_value']
r = wilcoxon['effect_size_r']
ci_lo, ci_hi = wilcoxon['ci_diff']

n_covered   = int(df_bl['covered'].sum())
n_uncovered = int((~df_bl['covered']).sum())
bl_cov_acc  = df_bl[df_bl['covered']]['correct'].mean()
kg_cov_acc  = df_kg[df_kg['covered']]['correct'].mean()
bl_unc_acc  = df_bl[~df_bl['covered']]['correct'].mean()
kg_unc_acc  = df_kg[~df_kg['covered']]['correct'].mean()

fixes_list  = [(i, t, bl, kg) for i, (t, bl, kg) in enumerate(zip(trues_bl, preds_bl, preds_kg)) 
               if bl != t and kg == t]
errors_list = [(i, t, bl, kg) for i, (t, bl, kg) in enumerate(zip(trues_bl, preds_bl, preds_kg)) 
               if bl == t and kg != t]

n_fixes  = len(fixes_list)
n_errors = len(errors_list)
net      = n_fixes - n_errors

per_net = sum(1 for _,t,_,_ in fixes_list if t in ('B-PER','I-PER')) - \
          sum(1 for _,t,_,_ in errors_list if t in ('B-PER','I-PER'))
loc_net = sum(1 for _,t,_,_ in fixes_list if t in ('B-LOC','I-LOC')) - \
          sum(1 for _,t,_,_ in errors_list if t in ('B-LOC','I-LOC'))
org_net = sum(1 for _,t,_,_ in fixes_list if t in ('B-ORG','I-ORG')) - \
          sum(1 for _,t,_,_ in errors_list if t in ('B-ORG','I-ORG'))



print(f"""
1. STATISTICAL CONTEXT
   Wilcoxon p={p:.4f} — no significant improvement at α=0.05
   Mean F1 delta: {diff:+.4f} (baseline {bl_mean:.4f} → KG {kg_mean:.4f})
   95% CI on difference: [{ci_lo:.4f}, {ci_hi:.4f}]
   Effect size r={r:.4f} (medium)

2. PREDICTION CHANGE ANALYSIS (seed42, token-level)
   KG fixes {n_fixes} baseline errors, introduces {n_errors} new errors (net {net:+d})
   PER: net {per_net:+d}  — KG helps person entity detection
   LOC: net {loc_net:+d}  — KG marginally helps location spans
   ORG: net {org_net:+d} — KG hurts organisation detection most

3. COVERAGE-STRATIFIED ANALYSIS
   KG-covered entities  (n={n_covered}):  baseline={bl_cov_acc:.3f}  KG={kg_cov_acc:.3f}  delta={kg_cov_acc-bl_cov_acc:+.3f}
   KG-uncovered entities (n={n_uncovered}): baseline={bl_unc_acc:.3f}  KG={kg_unc_acc:.3f}  delta={kg_unc_acc-bl_unc_acc:+.3f}

4. INTERPRETATION
   The coverage-stratified result is the most important finding:
   KG-covered entities do not show better accuracy than uncovered ones.
   This confirms that at 0.29% token-level coverage, the KG signal is
   too sparse to produce consistent entity-level gains. The TransE
   embeddings are not providing reliable disambiguation — the zero
   vectors for uncovered tokens and the noisy embeddings for covered
   ones both contribute to high variance across seeds.

   The ORG token-level net of {org_net:+d} is the clearest negative signal.
   ORG entities in this corpus are typically multi-token institutional
   names with low KG coverage, making them vulnerable to embedding noise.

   PER net {per_net:+d} is the only positive signal and aligns with the KG
   being parliamentary-focused — person entities are best represented.

5. DISSERTATION IMPLICATION
   These findings directly motivate Phase B: a broader, multi-domain KG
   with higher coverage (target >5% token-level) is needed before KG
   augmentation can produce reliable, statistically significant gains
   for Irish NER.
""")

XAI ANALYSIS SUMMARY

1. STATISTICAL CONTEXT
   Wilcoxon p=0.2344 — no significant improvement at α=0.05
   Mean F1 delta: +0.0097 (baseline 0.7344 → KG 0.7441)
   95% CI on difference: [-0.0099, 0.0294]
   Effect size r=0.3194 (medium)

2. PREDICTION CHANGE ANALYSIS (seed42, token-level)
   KG fixes 35 baseline errors, introduces 44 new errors (net -9)
   PER: net +4  — KG helps person entity detection
   LOC: net +2  — KG marginally helps location spans
   ORG: net -17 — KG hurts organisation detection most

3. COVERAGE-STRATIFIED ANALYSIS
   KG-covered entities  (n=25):  baseline=0.920  KG=0.840  delta=-0.080
   KG-uncovered entities (n=388): baseline=0.804  KG=0.791  delta=-0.013

4. INTERPRETATION
   The coverage-stratified result is the most important finding:
   KG-covered entities do not show better accuracy than uncovered ones.
   This confirms that at 0.29% token-level coverage, the KG signal is
   too sparse to produce consistent entity-level gains. The TransE
   embeddings 

The coverage-stratified result is actually stronger than expected. Covered entities (n=25) show lower accuracy with KG than without (0.840 vs 0.920, delta -0.080). This is a meaningful finding — the KG embeddings are actively hurting performance on the entities they were built to help. This is worth highlighting explicitly in the dissertation as evidence that embedding quality, not just coverage, is the bottleneck. A PER covered delta of -0.250 (n=4) is the most striking single number in the whole notebook.